<a href="https://colab.research.google.com/github/kolshaan/Hackathon-2026-UpsideDown/blob/Dev/Zenith_Tiger_Case_Study-Copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q litellm openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 11.0 MB/s eta 0:00:00


## **Test API Gateway Connect to Gemini 2.5 Flash Model**


In [ ]:
import openai

apikey = "sk-42HMv1aZopQFoiry1usRKg"  # Key generated from the ai-gateway
base_url = "https://api.ai-gateway.tigeranalytics.com"
model = "gemini-2.5-flash"  # Gemini 2.5 Flash model
user_query = "What is the capital of France?"  # The text we want to ask the model

client = openai.OpenAI(
    api_key=apikey,
    base_url=base_url
)

response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": user_query
        }
    ]
)

print(response)

print(response.choices[0].message.content)

ChatCompletion(id='CZp6afmWBrX_698PsNXPgAE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is **Paris**.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1769642503, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=7, total_tokens=15, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=None, text_tokens=7, image_tokens=None)), vertex_ai_grounding_metadata=[], vertex_ai_url_context_metadata=[], vertex_ai_safety_results=[], vertex_ai_citation_metadata=[])
The capital of France is **Paris**.


## **Read Source Data**

In [2]:
import pandas as pd

# Base raw GitHub URL
base_url = "https://raw.githubusercontent.com/farveznoufal/Hackathon-2026-UpsideDown/main/"

# Load CSVs into DataFrames
demand_forecast = pd.read_csv(base_url + "demand_forecast.csv")
current_inventory = pd.read_csv(base_url + "current_inventory.csv")
product_master = pd.read_csv(base_url + "product_master.csv")
shipping_matrix = pd.read_csv(base_url + "shipping_matrix.csv")
factory_shipments = pd.read_csv(base_url + "factory_shipments.csv")

# Preview data
print("demand forecast")
display(demand_forecast.head())


print("current_inventory")
display(current_inventory.head())


print("Product Master")
display(product_master.head())

print("Shipping Matrix")
display(shipping_matrix.head())

print("Factory Shipments")
display(factory_shipments.head())

demand forecast


,Date,Warehouse_ID,SKU_ID,Forecasted_Qty
0,2026-01-26,W01_New_Jersey,ZEN-101,52
1,2026-01-27,W01_New_Jersey,ZEN-101,51
2,2026-01-28,W01_New_Jersey,ZEN-101,52
3,2026-01-29,W01_New_Jersey,ZEN-101,32
4,2026-01-30,W01_New_Jersey,ZEN-101,49


current_inventory


,Date,Warehouse_ID,SKU_ID,On_Hand_Qty,Safety_Stock_Target
0,2026-01-26,W01_New_Jersey,ZEN-101,10,100
1,2026-01-26,W01_New_Jersey,ZEN-201,300,200
2,2026-01-26,W01_New_Jersey,ZEN-102,120,80
3,2026-01-26,W01_New_Jersey,ZEN-301,180,120
4,2026-01-26,W01_New_Jersey,ZEN-401,90,60


Product Master


,SKU,Name,Category,Weight_Kg,COGS,Selling_Price,Factory_Lead_Time_Days,Static_Safety_Stock
0,ZEN-101,Dark Choco Truffles,Chocolate,0.8,25.0,55.0,7,100
1,ZEN-201,Spicy Jalapeno Chips,Snacks,6.5,10.0,16.0,21,200
2,ZEN-102,Sea Salt Caramel Bar,Chocolate,1.2,18.0,32.0,14,80
3,ZEN-301,Gummy Bears Case,Candy,2.5,14.0,24.0,14,120
4,ZEN-401,Protein Bars Box,Health,1.0,22.0,45.0,10,60


Shipping Matrix


,Origin_Warehouse,Destination_Warehouse,Shipping_Cost_Per_Kg,Transit_Time_Days
0,W01_New_Jersey,W02_Chicago,1.2,3
1,W01_New_Jersey,W03_Atlanta,0.5,1
2,W01_New_Jersey,W04_Dallas,1.2,3
3,W01_New_Jersey,W05_Denver,2.5,5
4,W01_New_Jersey,W06_Los_Angeles,2.5,5


Factory Shipments


,PO_Number,SKU_ID,Destination_Warehouse,Qty_Ordered,Expected_Arrival_Date,Status
0,PO-FACTORY-001,ZEN-301,W07_Seattle,500,2026-01-28,In-Transit
1,PO-FACTORY-002,ZEN-101,W01_New_Jersey,1000,2026-02-09,Production


## **Test API Gateway Connect to Gemini 2.0 Flash Model**

In [20]:
from openai import OpenAI

from google.colab import userdata
apikey = userdata.get('secret1')
base_url = "https://api.ai-gateway.tigeranalytics.com"

client = OpenAI(
    api_key=apikey,
    base_url=base_url
)

response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
        {"role": "user", "content": "Say hello from Gemini 2.0 Flash"}
    ]
)

print(response.choices[0].message.content)

Hello from Gemini 2.0 Flash! It's a pleasure to be here. How can I help you today?



## **Agent 1 - The Watchdog**

In [24]:
# df_inventory and df_forecast are already loaded
inventory_data = current_inventory.to_csv(index=False)
forecast_data = demand_forecast.to_csv(index=False)

prompt = f"""
ROLE: You are 'The Watchdog', an Inventory Monitor Agent.
PURPOSE: Scans the warehouse network to identify where stock is critically low ("Distress") and where it is overflowing ("Excess").

DATA:
--- INVENTORY ---
{inventory_data}

---

TASK:
1. Identify "Distress" for all SKUs (where On_Hand_Qty < 80% of Safety_Stock_Target).
2. Identify "Excess" for all SKUs (where On_Hand_Qty > 20% greater than Safety_Stock_Target).
3. Pair them up where a location with 'Excess' can supply a location with 'Needs' for all possible combinations.

OUTPUT:
Return ONLY a JSON list of objects with this format:
[{{"SKU": "ID", "Needs": "Loc_A", "Has_Excess": "Loc_B"}}]
"""

In [25]:
response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
        {"role": "system", "content": "You are a supply chain analyst that only outputs valid JSON."},
        {"role": "user", "content": prompt}
    ],
    response_format={ "type": "json_object" } # This ensures Gemini returns valid JSON
)

In [26]:
watchdog_report = response.choices[0].message.content
print(watchdog_report)

[
  {
    "SKU": "ZEN-101",
    "Needs": "W01_New_Jersey",
    "Has_Excess": "W02_Chicago"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W01_New_Jersey"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W02_Chicago"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W03_Atlanta"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W04_Dallas"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W05_Denver"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W06_Los_Angeles"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W08_Miami"
  },
  {
    "SKU": "ZEN-301",
    "Needs": "W07_Seattle",
    "Has_Excess": "W09_St_Louis"
  }
]


## **Check Agent 1 - Output**

In [27]:
import json
import pandas as pd
from google.colab import data_table

# 1. Parse the JSON string into a Python list
report_data = json.loads(watchdog_report)

# 2. Convert to a DataFrame
report_df = pd.DataFrame(report_data)

# 3. Enable Colab's interactive table view
data_table.enable_dataframe_formatter()

# 4. Display the report
report_df

,SKU,Needs,Has_Excess
0,ZEN-101,W01_New_Jersey,W02_Chicago
1,ZEN-301,W07_Seattle,W01_New_Jersey
2,ZEN-301,W07_Seattle,W02_Chicago
3,ZEN-301,W07_Seattle,W03_Atlanta
4,ZEN-301,W07_Seattle,W04_Dallas
5,ZEN-301,W07_Seattle,W05_Denver
6,ZEN-301,W07_Seattle,W06_Los_Angeles
7,ZEN-301,W07_Seattle,W08_Miami
8,ZEN-301,W07_Seattle,W09_St_Louis


### **AGENT 2 - The Economist**


In [42]:
import json
import pandas as pd

# Convert dataframes to CSV strings (same pattern as Agent 1)

# Base raw GitHub URL
base_url = "https://raw.githubusercontent.com/farveznoufal/Hackathon-2026-UpsideDown/main/"

# Load CSVs into DataFrames
product_master_data = pd.read_csv(base_url + "product_master.csv")
shipping_matrix_data = pd.read_csv(base_url + "shipping_matrix.csv")

prompt = f"""
ROLE: You are 'The Economist', a Cost Optimization Agent.
PURPOSE: Decide whether it is cheaper to BUY inventory from a vendor or TRANSFER internally.

DATA:
--- IMBALANCE REPORT (from Agent 1) ---
{watchdog_report}

--- PRODUCT MASTER ---
{product_master_data}

--- DEMAND FORECAST ---
{forecast_data}

--- SHIPPING MATRIX ---
{shipping_matrix_data}
---

TASK:
1. For each imbalance:
   - Calculate Cost to BUY = COGS × qty
   - Calculate Cost to TRANSFER = Shipping_Cost_Per_Kg × Quantity
2. Compare Buy vs Transfer.
3. Recommend the cheaper option.
4. Clearly state savings and time advantage.
5. Recommend Transfer Quantity based on DEMAND FORECAST and IMBALANCE REPORT

OUTPUT:
Return ONLY a JSON list with this format:
[
  {{
    "SKU": "ID",
    "Decision": "TRANSFER or BUY",
    "From": "Loc_B",
    "To": "Loc_A",
    "Quantity": 50,
    "Buy_Cost": 500,
    "Transfer_Cost": 100,
    "Savings": 400,
    "Time_Saved_Days": 12
  }}
]
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a supply chain economist. You reason carefully but output ONLY valid JSON."
        },
        {"role": "user", "content": prompt}
    ],
    response_format={"type": "json_object"}
)

economist_raw = response.choices[0].message.content
print(economist_raw)


[
  {
    "SKU": "ZEN-101",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W01_New_Jersey",
    "Quantity": 52,
    "Buy_Cost": 1300.00,
    "Transfer_Cost": 49.92,
    "Savings": 1250.08,
    "Time_Saved_Days": 4
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W01_New_Jersey",
    "To": "W07_Seattle",
    "Quantity": 100,
    "Buy_Cost": 1400.00,
    "Transfer_Cost": 225.00,
    "Savings": 1175.00,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W07_Seattle",
    "Quantity": 100,
    "Buy_Cost": 1400.00,
    "Transfer_Cost": 300.00,
    "Savings": 1100.00,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W03_Atlanta",
    "To": "W07_Seattle",
    "Quantity": 100,
    "Buy_Cost": 1400.00,
    "Transfer_Cost": 300.00,
    "Savings": 1100.00,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER

##**Enforcement of Economist Decision (Agent 2)**

In [43]:
economist_decisions = json.loads(economist_raw)

final_decisions = []

for decision in economist_decisions:
    sku = decision["SKU"]
    qty = decision["Quantity"]
    origin = decision["From"]
    destination = decision["To"]

    # Buy cost
    product = product_master_data[product_master_data["SKU"] == sku].iloc[0]
    buy_cost = product["COGS"] * qty
    buy_time = product["Factory_Lead_Time_Days"]

    # Transfer cost
    route = shipping_matrix_data[
        (shipping_matrix_data["Origin_Warehouse"] == origin) &
        (shipping_matrix_data["Destination_Warehouse"] == destination)
    ].iloc[0]

    transfer_cost = route["Shipping_Cost_Per_Kg"] * qty
    transfer_time = route["Transit_Time_Days"]

    savings = buy_cost - transfer_cost

    print(
        f"[Economist Monologue] SKU {sku}: "
        f"Buy = ${buy_cost}, Transfer = ${transfer_cost}, "
        f"Savings = ${savings}"
    )

    final_decisions.append({
        "SKU": sku,
        "Decision": "TRANSFER" if savings > 0 else "BUY",
        "From": origin,
        "To": destination,
        "Quantity": qty,
        "Buy_Cost": buy_cost,
        "Transfer_Cost": transfer_cost,
        "Savings": savings,
        "Time_Saved_Days": buy_time - transfer_time
    })

final_decisions


[Economist Monologue] SKU ZEN-101: Buy = $1300.0, Transfer = $62.4, Savings = $1237.6
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $250.0, Savings = $1150.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $120.0, Savings = $1280.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $250.0, Savings = $1150.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $120.0, Savings = $1280.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $50.0, Savings = $1350.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $50.0, Savings = $1350.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $250.0, Savings = $1150.0
[Economist Monologue] SKU ZEN-301: Buy = $1400.0, Transfer = $120.0, Savings = $1280.0


[{'SKU': 'ZEN-101',
  'Decision': 'TRANSFER',
  'From': 'W02_Chicago',
  'To': 'W01_New_Jersey',
  'Quantity': 52,
  'Buy_Cost': np.float64(1300.0),
  'Transfer_Cost': np.float64(62.4),
  'Savings': np.float64(1237.6),
  'Time_Saved_Days': np.int64(4)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W01_New_Jersey',
  'To': 'W07_Seattle',
  'Quantity': 100,
  'Buy_Cost': np.float64(1400.0),
  'Transfer_Cost': np.float64(250.0),
  'Savings': np.float64(1150.0),
  'Time_Saved_Days': np.int64(9)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W02_Chicago',
  'To': 'W07_Seattle',
  'Quantity': 100,
  'Buy_Cost': np.float64(1400.0),
  'Transfer_Cost': np.float64(120.0),
  'Savings': np.float64(1280.0),
  'Time_Saved_Days': np.int64(11)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W03_Atlanta',
  'To': 'W07_Seattle',
  'Quantity': 100,
  'Buy_Cost': np.float64(1400.0),
  'Transfer_Cost': np.float64(250.0),
  'Savings': np.float64(1150.0),
  'Time_Saved_Days

In [44]:
from google.colab import data_table

df_final = pd.DataFrame(final_decisions)
data_table.enable_dataframe_formatter()
df_final


,SKU,Decision,From,To,Quantity,Buy_Cost,Transfer_Cost,Savings,Time_Saved_Days
0,ZEN-101,TRANSFER,W02_Chicago,W01_New_Jersey,52,1300.0,62.4,1237.6,4
1,ZEN-301,TRANSFER,W01_New_Jersey,W07_Seattle,100,1400.0,250.0,1150.0,9
2,ZEN-301,TRANSFER,W02_Chicago,W07_Seattle,100,1400.0,120.0,1280.0,11
3,ZEN-301,TRANSFER,W03_Atlanta,W07_Seattle,100,1400.0,250.0,1150.0,9
4,ZEN-301,TRANSFER,W04_Dallas,W07_Seattle,100,1400.0,120.0,1280.0,11
5,ZEN-301,TRANSFER,W05_Denver,W07_Seattle,100,1400.0,50.0,1350.0,13
6,ZEN-301,TRANSFER,W06_Los_Angeles,W07_Seattle,100,1400.0,50.0,1350.0,13
7,ZEN-301,TRANSFER,W08_Miami,W07_Seattle,100,1400.0,250.0,1150.0,9
8,ZEN-301,TRANSFER,W09_St_Louis,W07_Seattle,100,1400.0,120.0,1280.0,11


##**Agent 3 - The Executor to generate the final order**

In [45]:
executor_input = df_final.to_dict(orient="records")

In [46]:
executor_prompt = f"""
ROLE: You are "The Executor", a Supply Chain Fulfillment AI.

PURPOSE:
Convert approved optimization decisions into a professional Internal Transfer Order (ITO).

INPUT DATA:
--- DECISION RECORDS ---
{json.dumps(executor_input, indent=2)}
---

STRICT INSTRUCTIONS (VERY IMPORTANT):
1. EACH object in the input list represents ONE independent transfer.
2. You MUST copy the following fields EXACTLY AS PROVIDED for each transfer:
   - SKU
   - From
   - To
   - Quantity
   - Savings
   - Time_Saved_Days
3. DO NOT reuse, average, infer, or standardize quantities.
4. DO NOT perform any calculations.
5. DO NOT assume quantities are the same.
6. Preserve row-by-row integrity.

YOUR TASK:
1. Create an Executive Summary using the provided values.
2. Create one Transfer Order per input object.
3. Assign unique Transfer_Order_IDs in sequence (ITO-0001, ITO-0002, ...).
4. Use professional business language suitable for executive approval.

OUTPUT FORMAT:
Return ONLY valid JSON in the following structure:

{{
  "Executive_Summary": {{
    "Total_Transfers": <count>,
    "Total_Savings_USD": <sum of Savings>,
    "Average_Time_Saved_Days": <average>
  }},
  "Transfer_Orders": [
    {{
      "Transfer_Order_ID": "ITO-0001",
      "SKU": "<copied from input>",
      "From_Warehouse": "<copied from input>",
      "To_Warehouse": "<copied from input>",
      "Quantity": <copied from input>,
      "Estimated_Savings_USD": <copied from input>,
      "Estimated_Time_Saved_Days": <copied from input>,
      "Approval_Status": "Pending"
    }}
  ],
  "Business_Notes": "Short executive-friendly justification."
}}

OUTPUT RULES:
- JSON only
- No explanations
- No markdown
- No invented values
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a supply chain executor. You reason carefully but output ONLY valid JSON."
        },
        {"role": "user", "content": prompt}
    ],
    response_format={"type": "json_object"}
)

executor_raw = response.choices[0].message.content
print(executor_raw)


[
  {
    "SKU": "ZEN-101",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W01_New_Jersey",
    "Quantity": 52,
    "Buy_Cost": 1300.0,
    "Transfer_Cost": 49.92,
    "Savings": 1250.08,
    "Time_Saved_Days": 4
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W01_New_Jersey",
    "To": "W07_Seattle",
    "Quantity": 13,
    "Buy_Cost": 182.0,
    "Transfer_Cost": 81.25,
    "Savings": 100.75,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W07_Seattle",
    "Quantity": 5,
    "Buy_Cost": 70.0,
    "Transfer_Cost": 15.0,
    "Savings": 55.0,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W03_Atlanta",
    "To": "W07_Seattle",
    "Quantity": 7,
    "Buy_Cost": 98.0,
    "Transfer_Cost": 15.75,
    "Savings": 82.25,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W04_Dall

In [47]:
import json
import pandas as pd
from google.colab import data_table

executor_output = json.loads(executor_raw)

df_orders = pd.DataFrame(executor_output)
data_table.enable_dataframe_formatter()
df_orders




,SKU,Decision,From,To,Quantity,Buy_Cost,Transfer_Cost,Savings,Time_Saved_Days
0,ZEN-101,TRANSFER,W02_Chicago,W01_New_Jersey,52,1300.0,49.92,1250.08,4
1,ZEN-301,TRANSFER,W01_New_Jersey,W07_Seattle,13,182.0,81.25,100.75,11
2,ZEN-301,TRANSFER,W02_Chicago,W07_Seattle,5,70.0,15.00,55.00,11
3,ZEN-301,TRANSFER,W03_Atlanta,W07_Seattle,7,98.0,15.75,82.25,11
4,ZEN-301,TRANSFER,W04_Dallas,W07_Seattle,13,182.0,37.50,144.50,11
5,ZEN-301,TRANSFER,W05_Denver,W07_Seattle,9,126.0,22.50,103.50,9
6,ZEN-301,TRANSFER,W06_Los_Angeles,W07_Seattle,12,168.0,22.50,145.50,11
7,ZEN-301,TRANSFER,W08_Miami,W07_Seattle,13,182.0,48.75,133.25,11
8,ZEN-301,TRANSFER,W09_St_Louis,W07_Seattle,10,140.0,30.00,110.00,11
